In [20]:
import numpy as np 


with np.load("embeddings_main.npz") as data: 
    U = data["U"]
    V = data["V"]
    user_biases = data["user_biases"]
    movies_biases = data["movies_biases"]
    index_to_users = data["index_to_users"]
    index_to_movies = data["index_to_movies"]
    
users_to_index = {user:index for index,user in enumerate(index_to_users)}
movies_to_index = {movie:index for index,movie in enumerate(index_to_movies)}

# Create dummy user that like the "Avengers Infinity War"
dummy_user_arr = [[(movies_to_index[122912],5)]]

dummy_user_arr

[[(24728, 5)]]

In [21]:
# get top 10 movies 
with open("ml-32m/movies.csv", "r") as f: 
    lines = f.read()
    
    
movie_lines = lines.split("\n") 
movie_data = movie_lines[1:-1]
movie_cols = movie_lines[0]

movie_id_to_title = {}

for movie in movie_data:
    id = movie.split(",")[0]
    title = movie.split(",")[1]
    
    movie_id_to_title[id] = title 

In [22]:
N_ITER = 5
lamb = 0.09
gamma = 0.1
tau = 0.9
VERBOSITY = 1
K = 8

user_biases_ = np.zeros(len(dummy_user_arr))

U_ = np.random.normal(size = (len(dummy_user_arr),K))

for _ in range(N_ITER):

  # update bias
  for i, item_ratings in enumerate(dummy_user_arr):
    bias = 0
    item_counter = 0
    for (n,r) in item_ratings:
      bias += lamb * (r - movies_biases[n])
      item_counter += 1
    bias = bias / (lamb * item_counter + gamma)
    user_biases_[i] = bias

  for m, item_ratings in enumerate(dummy_user_arr):
    item_counter = 0
    u_comp1 = np.zeros([K,K])
    u_comp2 = np.zeros(K)
    for (n,r) in item_ratings:
      u_comp1 += lamb * np.outer(V[n],V[n])
      u_comp2 += lamb * (V[n] * (r - user_biases_[m] - movies_biases[n]))
      item_counter += 1
    u_comp1 = u_comp1 + tau * np.identity(K)
    U_[m] = np.linalg.solve(u_comp1,u_comp2)

In [23]:
dummy_u = U_[0]

ratings = []
for n,v in enumerate(V): 
    prod = dummy_u @ v + (movies_biases[n] * 0.01)
    ratings.append(prod)

In [24]:
movie_ranking = index_to_movies[np.argsort(ratings)[::-1]]

movie_ranking

array([122914., 122912.,  89745., ...,    743.,    135.,    520.],
      shape=(84432,))

In [ ]:
print("Top movies for dummy user that liked Avengers Infinity War: ")

for i in movie_ranking[:10]: 
    print(movie_id_to_title[str(int(i))])

Top movies for dummy user that liked Avengers Infinity War: 
Avengers: Infinity War - Part II (2019)
Avengers: Infinity War - Part I (2018)
"Avengers
Captain America: Civil War (2016)
Thor: Ragnarok (2017)
Captain America: The Winter Soldier (2014)
Avengers: Age of Ultron (2015)
Captain America: The First Avenger (2011)
Star Wars: Episode VII - The Force Awakens (2015)
Guardians of the Galaxy (2014)


## Finding Polarizing movies 

In [28]:
movie_distances = (V**2).sum(axis=1)

longest_distances_movies = np.argsort(movie_distances)[::-1]

for id in index_to_movies[longest_distances_movies][:10]: 
    print(movie_id_to_title[str(int(id))])

Dumb & Dumber (Dumb and Dumber) (1994)
Ace Ventura: Pet Detective (1994)
Natural Born Killers (1994)
Avengers: Infinity War - Part II (2019)
Avengers: Infinity War - Part I (2018)
Austin Powers: International Man of Mystery (1997)
American Pie (1999)
Star Wars: Episode IV - A New Hope (1977)
Babe (1995)
Borat: Cultural Learnings of America for Make Benefit Glorious Nation of Kazakhstan (2006)
